In [ ]:
import os
import glob
import zipfile

print(" Step 1: Processing Downloads folder for Kaggle ZIP file...")

download_folder = os.path.join(os.environ['USERPROFILE'], 'Downloads')

zip_files = glob.glob(os.path.join(download_folder, "*.zip"))

if not zip_files:
    print(" Error: No Zip File in Downloads.")
else:
    # Sabse latest file ko select kiya
    latest_zip_path = max(zip_files, key=os.path.getctime)
    print(f" Found ZIP file: {os.path.basename(latest_zip_path)}")
    
  
    target_directory = "D:/NIDS/2018"
    os.makedirs(target_directory, exist_ok=True)
    print(f" Target folder created or confirmed at: {target_directory}")
    
    # 4. Extraction Engine Start
    print("⏳ Extracting files... Isme data size ki wajah se 30-45 seconds lag sakte hain...")
    try:
        with zipfile.ZipFile(latest_zip_path, 'r') as zip_ref:
            zip_ref.extractall(target_directory)
        print(f"✅ SUCCESS: Saari parquet files '{target_directory}' folder me extract ho chuki hain!")
    except Exception as e:
        print(f" Error while extraction: {str(e)}")


In [ ]:
import os
import glob
import pandas as pd

# ========================================================
# PATH: Tumhare computer ka exact absolute 2018 data folder
# ========================================================
feature_path = r"D:/NIDS/2018"

# Parquet files list
files = glob.glob(os.path.join(feature_path, "*.parquet"))

print("=" * 60)
print("TOTAL FILES DETECTED IN 2018 DIRECTORY:", len(files))
print("=" * 60)

for file in files:
    print(os.path.basename(file))


In [ ]:
import os
import pandas as pd

print("=" * 70)
print("🔍 REAL-TIME ATTACK LABELS COUNT IN 2018 DATASETS")
print("=" * 70)

for file in files:
    
    temp_df = pd.read_parquet(file, columns=["Label"])
    
    print(f"\n File Name: {os.path.basename(file)}")
    print("-" * 50)
    print(temp_df["Label"].value_counts())
    print("-" * 50)


In [ ]:
# ==========================================
# COMBINE LABEL COUNTS FROM ALL FILES
# ==========================================

all_labels = []

for file in files:
    temp_df = pd.read_parquet(
        file,
        columns=["Label"]
    )

    all_labels.extend(
        temp_df["Label"].dropna().unique()
    )

print("=" * 70)
print("ALL UNIQUE LABELS")
print("=" * 70)

for label in sorted(set(all_labels)):
    print(label)

In [ ]:
# ==========================================
# SELECT ZERO-DAY ATTACK FAMILY
# ==========================================

ZERO_DAY_LABEL = "Bot"

print("Selected Zero-Day Attack:", ZERO_DAY_LABEL)

# Check how many samples belong to Zero-Day class
for file in files:

    temp_df = pd.read_parquet(
        file,
        columns=["Label"]
    )

    count = (temp_df["Label"] == ZERO_DAY_LABEL).sum()

    if count > 0:
        print(
            os.path.basename(file),
            "->",
            count,
            "samples"
        )

In [ ]:
# ==========================================
# EXPERIMENT 2 - ZERO-DAY DATA PREPARATION
# ==========================================

ZERO_DAY_LABEL = "Bot"

train_parts = []
zero_day_parts = []

for file in files:

    temp_df = pd.read_parquet(file)

    # Bot samples -> Zero-Day test
    bot_df = temp_df[
        temp_df["Label"] == ZERO_DAY_LABEL
    ].copy()

    # Everything except Bot -> Training pool
    non_bot_df = temp_df[
        temp_df["Label"] != ZERO_DAY_LABEL
    ].copy()

    if len(bot_df) > 0:
        zero_day_parts.append(bot_df)

    if len(non_bot_df) > 0:
        train_parts.append(non_bot_df)


# Combine
zero_day_df = pd.concat(
    zero_day_parts,
    ignore_index=True
)

train_pool_df = pd.concat(
    train_parts,
    ignore_index=True
)

print("=" * 70)
print("ZERO-DAY DATA PREPARATION")
print("=" * 70)

print("Training Pool Shape :", train_pool_df.shape)
print("Zero-Day Bot Shape  :", zero_day_df.shape)

print("\nBot samples in Zero-Day set:")
print(zero_day_df["Label"].value_counts())

print("\nLabels in Training Pool:")
print(train_pool_df["Label"].value_counts())

In [ ]:
# ==========================================
# STEP 5 - CREATE BINARY TARGET
# ==========================================

def create_target(df):
    df = df.copy()

    df["Target"] = (
        df["Label"]
        .str.upper()
        .ne("BENIGN")
        .astype(int)
    )

    return df


# Training target
train_pool_df = create_target(train_pool_df)

# Zero-Day target
zero_day_df = create_target(zero_day_df)


print("=" * 70)
print("TARGET DISTRIBUTION")
print("=" * 70)

print("\nTraining Pool:")
print(train_pool_df["Target"].value_counts())

print("\nZero-Day Set:")
print(zero_day_df["Target"].value_counts())

print("\nZero-Day Labels:")
print(zero_day_df["Label"].value_counts())

In [ ]:
# ==========================================
# STEP 6 - ZERO-DAY TRAIN / TEST PREPARATION
# ==========================================

from sklearn.model_selection import train_test_split
import numpy as np

# Copy data
known_df = train_pool_df.copy()
bot_df = zero_day_df.copy()

# Features and target
X_known = known_df.drop(columns=["Label", "Target"])
y_known = known_df["Target"]

X_bot = bot_df.drop(columns=["Label", "Target"])
y_bot = bot_df["Target"]

# Keep numeric features only
numeric_features = X_known.select_dtypes(
    include=["number"]
).columns

X_known = X_known[numeric_features]
X_bot = X_bot[numeric_features]

# Replace infinite values
X_known = X_known.replace(
    [np.inf, -np.inf],
    np.nan
)

X_bot = X_bot.replace(
    [np.inf, -np.inf],
    np.nan
)

# Use training medians for missing values
medians = X_known.median()

X_known = X_known.fillna(medians)
X_bot = X_bot.fillna(medians)

# Split known data
X_train, X_known_test, y_train, y_known_test = train_test_split(
    X_known,
    y_known,
    test_size=0.20,
    random_state=42,
    stratify=y_known
)

# Add completely unseen Bot attacks to test set
X_test = pd.concat(
    [X_known_test, X_bot],
    ignore_index=True
)

y_test = pd.concat(
    [y_known_test, y_bot],
    ignore_index=True
)

print("=" * 70)
print("ZERO-DAY TRAIN / TEST SETUP")
print("=" * 70)

print("Training shape :", X_train.shape)
print("Test shape     :", X_test.shape)

print("\nTraining target:")
print(y_train.value_counts())

print("\nTest target:")
print(y_test.value_counts())

print("\nBot samples in training:",
      len(X_train) if False else 0)

In [9]:
# ==========================================
# STEP 7 - ZERO-DAY MODEL TRAINING
# ==========================================

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import StandardScaler

# ------------------------------------------
# Random Forest
# ------------------------------------------

rf_zero_day = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_zero_day.fit(
    X_train,
    y_train
)

# ------------------------------------------
# Isolation Forest
# Train only on BENIGN traffic
# ------------------------------------------

X_train_benign = X_train[
    y_train == 0
]

scaler_zero_day = StandardScaler()

X_train_benign_scaled = scaler_zero_day.fit_transform(
    X_train_benign
)

X_test_scaled = scaler_zero_day.transform(
    X_test
)



KeyboardInterrupt: 

In [ ]:
isolation_zero_day = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

isolation_zero_day.fit(
    X_train_benign_scaled
)

print("=" * 70)
print("ZERO-DAY MODELS TRAINED")
print("=" * 70)

print("Random Forest trained successfully.")
print("Isolation Forest trained successfully.")
print("Bot samples used during training: 0")

In [ ]:
# ==========================================
# STEP 8 - ZERO-DAY EVALUATION + HYBRID
# ==========================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------
# Random Forest prediction
# ------------------------------------------

rf_zero_prob = rf_zero_day.predict_proba(
    X_test
)[:, 1]

rf_zero_pred = (
    rf_zero_prob >= 0.50
).astype(int)

# ------------------------------------------
# Isolation Forest prediction
# ------------------------------------------

iso_raw = isolation_zero_day.decision_function(
    X_test_scaled
)

# Lower decision_function = more anomalous
iso_anomaly_score = -iso_raw

# Convert anomaly score to binary prediction
iso_threshold = np.percentile(
    iso_anomaly_score,
    80
)

iso_zero_pred = (
    iso_anomaly_score >= iso_threshold
).astype(int)

# ------------------------------------------
# Hybrid Risk Score
# ------------------------------------------

# Normalize Isolation Forest score
iso_min = iso_anomaly_score.min()
iso_max = iso_anomaly_score.max()

iso_normalized = (
    (iso_anomaly_score - iso_min) /
    (iso_max - iso_min + 1e-10)
)

hybrid_zero_risk = (
    0.5 * rf_zero_prob +
    0.5 * iso_normalized
)

hybrid_zero_pred = (
    hybrid_zero_risk >= 0.50
).astype(int)

# ------------------------------------------
# Evaluation Function
# ------------------------------------------

def evaluate_zero_day(name, y_true, y_pred):

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(
        y_true, y_pred, zero_division=0
    ))
    print("Recall   :", recall_score(
        y_true, y_pred, zero_division=0
    ))
    print("F1 Score :", f1_score(
        y_true, y_pred, zero_division=0
    ))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            zero_division=0
        )
    )


# Evaluate all three
evaluate_zero_day(
    "Random Forest - Zero-Day",
    y_test,
    rf_zero_pred
)

evaluate_zero_day(
    "Isolation Forest - Zero-Day",
    y_test,
    iso_zero_pred
)

evaluate_zero_day(
    "Hybrid - Zero-Day",
    y_test,
    hybrid_zero_pred
)